# CPE 342 Term Project — Task 5: Account Security Monitoring (Unsupervised Anomaly Detection)
**Course:** CPE 342 Machine Learning (Semester 1/2026) — KMUTT  
**System Component:** Karena Thailand Player Intelligence System — Account Takeover, Bot Ring & VPN Fraud Detection  
**Task Category:** Unsupervised Anomaly Detection (`is_anomaly`: `0` = Normal, `1` = Anomalous; **No Labeled Training Data — `test.csv` only**)  
**Primary Evaluation Metric:** **$F_3$ Score** ($F_3 = \frac{10 \cdot P \cdot R}{9P + R}$, weighting Recall $3\times$ higher than Precision)  

### Official Competition Rule Compliance
* **Data Usage & Model Development (Rules 2 & 3):** Because Task 5 is an unsupervised anomaly detection challenge where only `test.csv` is provided, all detectors (`IsolationForest`, `ECOD`, `LOF`, and the `TabularAutoencoder` initialized from random weights) are trained **100% from scratch** on the provided `task5/data/test.csv` dataset with zero external data or pre-trained weights.
* **Portable Execution:** Supports **Google Colab GPU**, **Kaggle Kernels**, and **Local Workspace**.

## 1. Environment Setup & Path Resolution

In [ ]:
import os
import sys
import warnings
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.decomposition import PCA
from scipy.stats import rankdata

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

BASE_DIR = Path.cwd()
if (BASE_DIR / "project" / "task5").exists():
    TASK_DIR = BASE_DIR / "project" / "task5"
elif (BASE_DIR / "task5").exists():
    TASK_DIR = BASE_DIR / "task5"
else:
    TASK_DIR = BASE_DIR

DATA_DIR = TASK_DIR / "data"
MODELS_DIR = TASK_DIR / "models"
PREDICTIONS_DIR = TASK_DIR / "predictions"
ASSETS_DIR = TASK_DIR / "assets"

for d in [DATA_DIR, MODELS_DIR, PREDICTIONS_DIR, ASSETS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial"]
plt.rcParams["axes.unicode_minus"] = False

print(f"Task 5 Directory: {TASK_DIR}")

## 2. Data Loading & Longitudinal Feature Inspection (`test.csv`: 25,889 Rows $\times$ 123 Columns)
Inspection of `task5/data/test.csv` and `datacard.txt`:
* Contains **25,889 player accounts** and **122 predictive features** (no `train.csv` exists for this unsupervised task).
* Features are organized into two structural groups:
  1. **Longitudinal Time-Series Features (29 behavioral metrics $\times$ 4 sequential time periods `_1, _2, _3, _4` = 116 features):** e.g., `login_count`, `login_lat`, `login_lon`, `location_changes`, `device_count`, `ip_hash_entropy`, `skill_rating`, `kda_mean`, `accuracy_mean`, `reaction_time_mean`, `purchase_amount_mean`, `refund_count`, `currency_transfers`, `friend_removes`, `guild_changes`.
  2. **Lifetime & Security Risk Flags (6 features):** `account_age_days`, `behavior_stability`, `password_changed`, `vpn_usage`, `suspicious_login_time`, `mass_item_sale`.

In [ ]:
test_df = pd.read_csv(DATA_DIR / "test.csv")
ID_COL = "player_id"
RAW_FEATURES = [c for c in test_df.columns if c != ID_COL]

print(f"Task 5 Test Shape    : {test_df.shape}")
print(f"Total Missing Values : {test_df[RAW_FEATURES].isna().sum().sum():,}")

risk_flags = ["password_changed", "vpn_usage", "suspicious_login_time", "mass_item_sale"]
print("\nSecurity Risk Flag Summaries:")
display(test_df[risk_flags + ["behavior_stability", "account_age_days"]].describe().round(3))

## 3. Spatio-Temporal & Domain Security Feature Engineering
To detect **Account Takeovers (hacked accounts)**, **24/7 Bot Farms**, and **VPN Regional Pricing Arbitrage**, we engineer physical and temporal anomaly features:
1. **Haversine Geodesic Velocity (km across periods):** Computes exact great-circle surface distance between consecutive login coordinates `(login_lat_t, login_lon_t)` and `(login_lat_{t+1}, login_lon_{t+1})` to capture **Impossible Travel**.
2. **Temporal Surge & Period-4 Jump (`_4 - mean(_1,_2,_3)` and temporal `std`):** Compromised or sold accounts exhibit abrupt shifts in `skill_rating`, `kda_mean`, `device_count`, `ip_hash_entropy`, `currency_transfers`, and `friend_removes` during Period 4.
3. **Domain Security Risk Composite:** Aggregates explicit tripwires (`vpn_usage`, `password_changed`, `suspicious_login_time`, `mass_item_sale`, and `1 - behavior_stability`).

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2.0)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2.0)**2
    return 2.0 * r * np.arcsin(np.sqrt(np.clip(a, 0.0, 1.0)))

def engineer_task5_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df[RAW_FEATURES].copy()
    
    # 1. Haversine geodesic distance across consecutive periods (1->2, 2->3, 3->4)
    d12 = haversine_km(out["login_lat_1"], out["login_lon_1"], out["login_lat_2"], out["login_lon_2"])
    d23 = haversine_km(out["login_lat_2"], out["login_lon_2"], out["login_lat_3"], out["login_lon_3"])
    d34 = haversine_km(out["login_lat_3"], out["login_lon_3"], out["login_lat_4"], out["login_lon_4"])
    
    out["geo_dist_12_km"] = d12
    out["geo_dist_23_km"] = d23
    out["geo_dist_34_km"] = d34
    out["geo_dist_max_km"] = np.nanmax(np.vstack([d12, d23, d34]), axis=0)
    out["geo_dist_total_km"] = np.nansum(np.vstack([d12, d23, d34]), axis=0)
    
    # 2. Longitudinal Trajectory Deltas (Period 4 vs Period 1) & Temporal Std
    longitudinal_prefixes = [
        "login_count", "location_changes", "device_count", "ip_hash_entropy",
        "login_time_variance", "session_duration_mean", "play_pattern_consistency",
        "activity_level", "skill_rating", "skill_trajectory", "kda_mean",
        "accuracy_mean", "reaction_time_mean", "performance_stability",
        "purchase_frequency", "purchase_amount_mean", "refund_count",
        "currency_transfers", "friend_adds", "friend_removes",
        "communication_frequency", "guild_changes"
    ]
    for pref in longitudinal_prefixes:
        cols = [f"{pref}_1", f"{pref}_2", f"{pref}_3", f"{pref}_4"]
        vals = out[cols].values
        out[f"{pref}_delta_4_1"] = out[f"{pref}_4"] - out[f"{pref}_1"]
        out[f"{pref}_Jump_4_vs_mean123"] = out[f"{pref}_4"] - np.nanmean(vals[:, :3], axis=1)
        out[f"{pref}_temporal_std"] = np.nanstd(vals, axis=1)
        out[f"{pref}_temporal_max"] = np.nanmax(vals, axis=1)
        
    # 3. Account Takeover / Dumping Composite Indicators
    out["security_flag_sum"] = (
        out["password_changed"].fillna(0)
        + out["vpn_usage"].fillna(0)
        + out["suspicious_login_time"].fillna(0)
        + out["mass_item_sale"].fillna(0)
    )
    out["instability_index"] = 1.0 - out["behavior_stability"].fillna(out["behavior_stability"].median())
    out["missing_count"] = out[RAW_FEATURES].isna().sum(axis=1)
    return out

X_fe = engineer_task5_features(test_df)
print(f"Engineered Task 5 Feature Matrix: {X_fe.shape}")

## 4. Multi-Paradigm Unsupervised Anomaly Ensemble (Isolation Forest + ECOD + LOF + Deep Autoencoder)
We combine 5 complementary unsupervised anomaly detection families via **Percentile Rank Averaging**:
1. **Isolation Forest (Random Tree Partitioning):** Isolates global multivariate outliers via random hyperplanes.
2. **ECOD (Empirical Cumulative Distribution Outlier Detection):** Parameter-free non-parametric tail probability estimation across all dimensions.
3. **Local Outlier Factor (Density-Based LOF):** Detects local neighborhood density deviations in PCA subspace.
4. **Deep Reconstruction Autoencoder (PyTorch Neural Network trained from scratch):** Captures non-linear manifold reconstruction errors ($\|x - \hat{x}\|_2^2$).
5. **Domain Security Risk Rank:** Explicit tail scoring on high-risk account takeover indicators.

In [ ]:
# Preprocess: Median Imputation + RobustScaler (resilient to extreme outliers)
imputer = SimpleImputer(strategy="median")
scaler = RobustScaler()
X_imp = imputer.fit_transform(X_fe)
X_scaled = scaler.fit_transform(X_imp)
X_scaled = np.clip(X_scaled, -15.0, 15.0)

# 1. Isolation Forest Ensemble (multiple seeds & feature sub-samples)
iforest_scores = np.zeros(len(X_scaled))
for i, max_f in enumerate([0.5, 0.7, 1.0]):
    iso = IsolationForest(
        n_estimators=400,
        max_samples=2048,
        max_features=max_f,
        random_state=SEED + i * 11,
        n_jobs=-1
    )
    iso.fit(X_scaled)
    iforest_scores += (-iso.score_samples(X_scaled)) / 3.0

# 2. ECOD (Empirical Cumulative Distribution Outlier Detection - Parameter-free tail probability)
def compute_ecod_scores(X: np.ndarray) -> np.ndarray:
    n, d = X.shape
    left_p = np.zeros_like(X)
    right_p = np.zeros_like(X)
    for j in range(d):
        r_left = rankdata(X[:, j], method="max") / (n + 1.0)
        r_right = rankdata(-X[:, j], method="max") / (n + 1.0)
        left_p[:, j] = -np.log(r_left)
        right_p[:, j] = -np.log(r_right)
    return np.maximum(left_p, right_p).sum(axis=1)

ecod_scores = compute_ecod_scores(X_scaled)

# 3. Local Outlier Factor (Density-based on top 30 PCA components)
pca_30 = PCA(n_components=30, random_state=SEED)
X_pca30 = pca_30.fit_transform(X_scaled)
lof = LocalOutlierFactor(n_neighbors=35, novelty=False, n_jobs=-1)
lof.fit(X_pca30)
lof_scores = -lof.negative_outlier_factor_

# 4. Deep Reconstruction Autoencoder (PyTorch)
class TabularAutoencoder(nn.Module):
    def __init__(self, in_dim: int):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.BatchNorm1d(128),
            nn.SiLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 48),
            nn.BatchNorm1d(48),
            nn.SiLU(),
            nn.Linear(48, 20),
        )
        self.decoder = nn.Sequential(
            nn.Linear(20, 48),
            nn.BatchNorm1d(48),
            nn.SiLU(),
            nn.Linear(48, 128),
            nn.BatchNorm1d(128),
            nn.SiLU(),
            nn.Linear(128, in_dim),
        )
    def forward(self, x):
        return self.decoder(self.encoder(x))

ae_model = TabularAutoencoder(X_scaled.shape[1])
ae_opt = torch.optim.AdamW(ae_model.parameters(), lr=2e-3, weight_decay=1e-4)
ae_loader = DataLoader(TensorDataset(torch.tensor(X_scaled, dtype=torch.float32)), batch_size=256, shuffle=True)

ae_model.train()
for epoch in range(15):
    for (bx,) in ae_loader:
        ae_opt.zero_grad()
        recon = ae_model(bx)
        loss = torch.mean((bx - recon) ** 2)
        loss.backward()
        ae_opt.step()

ae_model.eval()
with torch.no_grad():
    full_t = torch.tensor(X_scaled, dtype=torch.float32)
    ae_scores = torch.mean((full_t - ae_model(full_t)) ** 2, dim=1).numpy()

# 5. Domain Security Risk Score (Explicit takeover/bot indicators)
domain_cols = [
    "security_flag_sum", "instability_index", "geo_dist_max_km",
    "device_count_temporal_max", "ip_hash_entropy_temporal_max",
    "currency_transfers_temporal_max", "refund_count_temporal_max",
    "friend_removes_temporal_max", "skill_trajectory_temporal_std"
]
domain_ranks = np.mean([rankdata(X_fe[c].fillna(X_fe[c].median())) / len(X_fe) for c in domain_cols], axis=0)

# Rank-normalize all 5 detector families into [0, 1] and fuse (Soft Voting + Max Tail Boost)
r_iforest = rankdata(iforest_scores) / len(X_fe)
r_ecod = rankdata(ecod_scores) / len(X_fe)
r_lof = rankdata(lof_scores) / len(X_fe)
r_ae = rankdata(ae_scores) / len(X_fe)
r_dom = rankdata(domain_ranks) / len(X_fe)

soft_vote_score = 0.25 * r_iforest + 0.20 * r_ecod + 0.15 * r_lof + 0.20 * r_ae + 0.20 * r_dom
hard_votes_top20 = (
    (r_iforest >= 0.80).astype(int)
    + (r_ecod >= 0.80).astype(int)
    + (r_lof >= 0.80).astype(int)
    + (r_ae >= 0.80).astype(int)
    + (r_dom >= 0.80).astype(int)
)

print("Unsupervised Detector Pairwise Rank Correlations:")
corr_mat = pd.DataFrame({
    "IsolationForest": r_iforest,
    "ECOD": r_ecod,
    "LOF": r_lof,
    "Autoencoder": r_ae,
    "DomainRisk": r_dom,
    "Ensemble_SoftVote": soft_vote_score
}).corr()
display(corr_mat.round(3))

## 5. $F_3$-Optimal Contamination Thresholding, PCA Anomaly Visualization & Predictions
Because the **$F_3$ Score** ($F_3 = \frac{10 \cdot P \cdot R}{9P + R}$) penalizes False Negatives **9 times more heavily** than False Positives ($eta^2 = 9$), the anomaly decision boundary must prioritize high Recall. We calibrate a recall-oriented contamination threshold combining soft-rank quantiles and multi-detector consensus voting, saving 2D PCA visualizations to `task5/assets/` and predictions to `task5/predictions/test_predictions.csv`.

In [ ]:
# Contamination cutoff tuned for F3 (Recall weighted 3x over Precision)
# Natural break / high-recall anomaly quantile (top 22% suspicious accounts OR >= 3 hard detector votes)
CONTAMINATION_RATE = 0.22
cutoff = np.quantile(soft_vote_score, 1.0 - CONTAMINATION_RATE)
pred_anomaly = ((soft_vote_score >= cutoff) | (hard_votes_top20 >= 3)).astype(int)

print(f"F3-Tuned Anomaly Flagged Count : {pred_anomaly.sum():,} / {len(pred_anomaly):,} ({pred_anomaly.mean()*100:.2f}%)")

# Visualize Anomaly Score Distribution & 2D PCA Projection
pca_2 = PCA(n_components=2, random_state=SEED).fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
axes[0].hist(soft_vote_score[pred_anomaly == 0], bins=45, color="#2563EB", alpha=0.8, label="Normal (0)")
axes[0].hist(soft_vote_score[pred_anomaly == 1], bins=45, color="#DC2626", alpha=0.8, label="Anomalous (1)")
axes[0].axvline(cutoff, color="#111827", linestyle="--", lw=2, label=f"F3 Cutoff ({cutoff:.3f})")
axes[0].set_title("Task 5: Unsupervised Ensemble Anomaly Score Distribution", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Ensemble Soft-Vote Anomaly Rank Score")
axes[0].set_ylabel("Number of Accounts")
axes[0].legend()
axes[0].grid(alpha=0.3)

idx_sample = np.random.choice(len(pca_2), size=6000, replace=False)
axes[1].scatter(pca_2[idx_sample][pred_anomaly[idx_sample] == 0, 0],
                pca_2[idx_sample][pred_anomaly[idx_sample] == 0, 1],
                c="#2563EB", s=10, alpha=0.35, label="Normal (0)")
axes[1].scatter(pca_2[idx_sample][pred_anomaly[idx_sample] == 1, 0],
                pca_2[idx_sample][pred_anomaly[idx_sample] == 1, 1],
                c="#DC2626", s=14, alpha=0.65, label="Flagged Anomaly (1)")
axes[1].set_title("Task 5: 2D PCA Projection of Normal vs Flagged Accounts", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Principal Component 1")
axes[1].set_ylabel("Principal Component 2")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(ASSETS_DIR / "anomaly-score-and-pca-projection.png", dpi=300, bbox_inches="tight")
plt.show()

# Save anomaly profile comparison table
profile_cols = ["security_flag_sum", "instability_index", "geo_dist_max_km", "device_count_4", "ip_hash_entropy_4", "currency_transfers_4", "refund_count_4"]
profile_df = X_fe[profile_cols].groupby(pred_anomaly).mean().T
profile_df.columns = ["Normal_Mean (0)", "Anomalous_Mean (1)"]
profile_df["Anomaly_to_Normal_Ratio"] = profile_df["Anomalous_Mean (1)"] / (profile_df["Normal_Mean (0)"].abs() + 1e-5)
profile_df.to_csv(ASSETS_DIR / "anomaly-cluster-profile.csv")
display(profile_df.round(3))

# Save Test Predictions (including multiple contamination thresholds for easy tuning)
test_out = pd.DataFrame({
    "player_id": test_df[ID_COL],
    "anomaly_score": soft_vote_score.round(6),
    "hard_votes": hard_votes_top20,
    "task5": pred_anomaly,
    "task5_top15pct": (soft_vote_score >= np.quantile(soft_vote_score, 0.85)).astype(int),
    "task5_top20pct": (soft_vote_score >= np.quantile(soft_vote_score, 0.80)).astype(int),
    "task5_top25pct": (soft_vote_score >= np.quantile(soft_vote_score, 0.75)).astype(int),
})
test_out.to_csv(PREDICTIONS_DIR / "test_predictions.csv", index=False)
print(f"Saved Task 5 predictions to: {PREDICTIONS_DIR / 'test_predictions.csv'} ({len(test_out):,} rows)")